In [1]:
"""
================================================================================
程序1：树模型组完整流程（阶段0-8）- 优化版
================================================================================
专注于树模型组：RandomForest / XGBoost / GradientBoosting / ExtraTrees / HistGB

✅ 主要优化：
1. 放宽筛选阈值（提高合格模型数量）
2. CV策略改为标准KFold（降低标准差，提升效率）
3. 优化RF/XGB/GB超参数（降低过拟合）
4. 清理非树模型导入
5. 新增训练集MAE和RMSE的计算、打印和保存 ← v2.2新增

版本：v2.2 - 训练集MAE/RMSE完善版
日期：2024-11
================================================================================
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, RepeatedKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_squared_log_error

# ✅ 只保留树模型相关的导入
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor, ExtraTreesRegressor)
from xgboost import XGBRegressor
from sklearn.base import clone

# 保留必要的工具类导入
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mutual_info_score
from scipy.stats import pearsonr, spearmanr, rankdata
import os
import warnings
import pickle
import time
import itertools
from collections import Counter
import multiprocessing
from joblib import Parallel, delayed

try:
    from dcor import distance_correlation
    DCOR_AVAILABLE = True
except ImportError:
    DCOR_AVAILABLE = False
    print("Warning: dcor not available, distance correlation will not be calculated")

warnings.filterwarnings('ignore')

n_cores = multiprocessing.cpu_count()
print(f"检测到 {n_cores} 个CPU核心")

# ==================== 配置参数 ====================
CONFIG = {
    'data_file': "D:\\博一下\\第一章主动学习\\Nb-Si数据库10.18-26个-4种参数-成分-性能-Si小于10的全部去掉.xlsx",
    'output_dir': "D:\\博一下\\第一章主动学习\\树模型组_结果11.9_优化版-4.22",
    'target_col': 'KQ',
    'exclude_cols': ['ID', 'Sample_Name', 'Notes'],

    'stage1_target': 180,
    'stage2_target': 60,
    'stage4_tree_target': 15,

    # ✅ 修改1：放宽筛选阈值
    'min_r2_threshold': 0.5,
    'max_r2_threshold': 0.95,
    'max_generalization_gap': 0.2,

    'n_repeat_seeds': 10,
    'base_random_state': 2023,
    'stability_penalty': 0.1,

    'random_state': 2023,
    'test_size': 0.2,
    'n_jobs': 3,
    'stage6_n_jobs': 6,

    'randomization_strategy': 'random_state'
}

# ✅ 修改2：CV策略改为标准KFold
CONFIG['cv_strategy'] = {
    'stage5': {
        'method': 'kfold',
        'n_splits': 5,
        'shuffle': True,
        'random_state': 2023
    },
    'stage6': {
        'method': 'kfold',
        'n_splits': 5,
        'shuffle': True,
        'random_state': 2023
    },
    'stage8': {
        'method': 'kfold',
        'n_splits': 5,
        'shuffle': True,
        'random_state': 2023
    }
}

CONFIG['feature_tiers'] = {
    'S': {
        'features': ['Nb', 'Si', 'VEC', 'δ'],
        'weight': 1.5,
        'description': 'S级(核心特征)'
    },
    'A': {
        'features': [
            'x', 'Δx', 'ΔHmix', 'ΔSmix', 'ΔG', 'Ω', 'Λ', 'Tm', 'ΔTm',
            'ρ', 'r', 'am', 'Δa', 'D.χ', 'D.r', 'Ec',
            'Pugh_modulus_ratio', 'enthalpy_entropy_competition',
            'phase_stability_index', 'Nb_Si_ratio', 'eutectic_distance',
            'VEC_Omega_coupling'
        ],
        'weight': 1.2,
        'description': 'A级(重要特征)'
    },
    'B': {
        'weight': 1.0,
        'description': 'B级(标准特征)'
    }
}

CONFIG['stage1'] = {
    'unique_ratio_threshold': 0.05,
    'high_collinearity_threshold': 0.98,
}

CONFIG['stage2'] = {
    'weights': {
        'pearson': 0.45,
        'mi': 0.35,
        'discriminative': 0.20
    },
    'weights_with_dcor': {
        'pearson': 0.35,
        'mi': 0.30,
        'dcor': 0.20,
        'discriminative': 0.15
    },
    'tier_multipliers': {
        'S': 1.5,
        'A': 1.2,
        'B': 1.0
    },
    'mi_discretization': {
        'method': 'quantile',
        'n_bins': 10,
        'strategy': 'discrete'
    }
}

CONFIG['stage5'] = {
    'k_range': [4, 5, 6, 7],
    'top_n_per_k': 5,
    'n_jobs_parallel': 5
}

# 创建输出目录
os.makedirs(CONFIG['output_dir'], exist_ok=True)
predictions_dir = os.path.join(CONFIG['output_dir'], 'detailed_predictions')
os.makedirs(predictions_dir, exist_ok=True)
top100_dir = os.path.join(CONFIG['output_dir'], 'top100_models')
os.makedirs(top100_dir, exist_ok=True)
top100_predictions_dir = os.path.join(top100_dir, 'predictions')
os.makedirs(top100_predictions_dir, exist_ok=True)
top100_models_dir = os.path.join(top100_dir, 'model_objects')
os.makedirs(top100_models_dir, exist_ok=True)
top30_dir = os.path.join(CONFIG['output_dir'], 'top30_models')
os.makedirs(top30_dir, exist_ok=True)
top30_predictions_dir = os.path.join(top30_dir, 'predictions')
os.makedirs(top30_predictions_dir, exist_ok=True)
top30_models_dir = os.path.join(top30_dir, 'model_objects')
os.makedirs(top30_models_dir, exist_ok=True)
stage8_dir = os.path.join(CONFIG['output_dir'], 'stage8_ranking_consistency')
os.makedirs(stage8_dir, exist_ok=True)

# ==================== 数据清洗函数 ====================
def check_and_remove_invalid_values(X, y, data_name="数据"):
    """检查并移除包含 NaN 或 Inf/-Inf 的样本"""
    print(f"\n  检查{data_name}中的无效值...")

    n_samples_before = X.shape[0]

    nan_mask_x = ~np.isnan(X).any(axis=1)
    nan_mask_y = ~np.isnan(y)
    nan_removed = n_samples_before - np.sum(nan_mask_x & nan_mask_y)

    if nan_removed > 0:
        print(f"    ⚠️  发现 {nan_removed} 个含NaN的样本")

    inf_mask_x = ~np.isinf(X).any(axis=1)
    inf_mask_y = ~np.isinf(y)
    inf_removed = n_samples_before - np.sum(inf_mask_x & inf_mask_y)

    if inf_removed > 0:
        print(f"    ⚠️  发现 {inf_removed} 个含Inf/-Inf的样本")

    valid_mask = nan_mask_x & nan_mask_y & inf_mask_x & inf_mask_y

    X_clean = X[valid_mask]
    y_clean = y[valid_mask]
    valid_indices = np.where(valid_mask)[0]

    n_samples_after = X_clean.shape[0]
    n_removed_total = n_samples_before - n_samples_after

    if n_removed_total > 0:
        print(f"    ✅ 移除 {n_removed_total} 个无效样本 "
              f"({n_samples_before} → {n_samples_after})")
    else:
        print(f"    ✅ 数据clean，无无效值")

    return X_clean, y_clean, valid_indices


# ==================== 辅助函数 ====================
def create_stratify_bins(y, n_bins=5):
    """创建分层标签"""
    print(f"\n  创建分层标签({n_bins}个区间):")
    percentiles = np.linspace(0, 100, n_bins + 1)
    bin_edges = np.percentile(y, percentiles)
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf
    stratify_labels = np.digitize(y, bin_edges) - 1

    for i in range(n_bins):
        mask = stratify_labels == i
        count = np.sum(mask)
        if count > 0:
            y_in_bin = y[mask]
            print(f"    区间 {i+1}: KQ范围 [{y_in_bin.min():.2f}, {y_in_bin.max():.2f}], "
                  f"样本数={count}, 均值={y_in_bin.mean():.2f}")

    return stratify_labels

def assign_feature_tiers(features_name):
    """为所有特征分配等级标签"""
    print("\n==================== 特征分级标记 ====================")

    feature_tier_map = {}
    s_tier_features = CONFIG['feature_tiers']['S']['features']
    a_tier_features = CONFIG['feature_tiers']['A']['features']

    s_count = 0
    a_count = 0
    b_count = 0

    for feat in features_name:
        if feat in s_tier_features:
            feature_tier_map[feat] = 'S'
            s_count += 1
        elif feat in a_tier_features:
            feature_tier_map[feat] = 'A'
            a_count += 1
        else:
            feature_tier_map[feat] = 'B'
            b_count += 1

    print(f"\n特征分级统计:")
    print(f"  S级: {s_count}个 - {CONFIG['feature_tiers']['S']['description']}")
    print(f"  A级: {a_count}个 - {CONFIG['feature_tiers']['A']['description']}")
    print(f"  B级: {b_count}个 - {CONFIG['feature_tiers']['B']['description']}")

    return feature_tier_map

def calculate_comprehensive_metrics(y_true, y_pred):
    """计算全面的回归指标"""
    try:
        r2 = r2_score(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)

        y_true_nonzero = np.where(np.abs(y_true) < 1e-6, 1e-6, y_true)
        mape = np.mean(np.abs((y_true - y_pred) / y_true_nonzero)) * 100

        y_true_pos = np.where(y_true <= 0, 1e-6, y_true)
        y_pred_pos = np.where(y_pred <= 0, 1e-6, y_pred)
        rmsle = np.sqrt(mean_squared_log_error(y_true_pos, y_pred_pos))

        if np.isnan(r2) or np.isinf(r2):
            r2 = -float('inf')
        if np.isnan(rmse) or np.isinf(rmse):
            rmse = float('inf')
        if np.isnan(mae) or np.isinf(mae):
            mae = float('inf')
        if np.isnan(mape) or np.isinf(mape):
            mape = float('inf')
        if np.isnan(rmsle) or np.isinf(rmsle):
            rmsle = float('inf')

        return {
            'r2': r2, 'rmse': rmse, 'mae': mae,
            'mape': mape, 'rmsle': rmsle
        }
    except:
        return {
            'r2': -float('inf'), 'rmse': float('inf'), 'mae': float('inf'),
            'mape': float('inf'), 'rmsle': float('inf')
        }

def save_detailed_predictions(model_key, y_train, pred_train, y_test, pred_test,
                            train_indices, test_indices, timestamp):
    """保存模型的详细预测结果"""
    train_results = pd.DataFrame({
        'Dataset': ['Training'] * len(y_train),
        'Dataset_Code': [1] * len(y_train),
        'Sample_Index': train_indices + 1,
        'Experimental_KQ': y_train,
        'Predicted_KQ': pred_train,
        'Error': pred_train - y_train,
        'Absolute_Error': np.abs(pred_train - y_train),
        'Relative_Error_Percent': np.abs(pred_train - y_train) / np.abs(y_train) * 100
    })

    test_results = pd.DataFrame({
        'Dataset': ['Test'] * len(y_test),
        'Dataset_Code': [2] * len(y_test),
        'Sample_Index': test_indices + 1,
        'Experimental_KQ': y_test,
        'Predicted_KQ': pred_test,
        'Error': pred_test - y_test,
        'Absolute_Error': np.abs(pred_test - y_test),
        'Relative_Error_Percent': np.abs(pred_test - y_test) / np.abs(y_test) * 100
    })

    all_results = pd.concat([train_results, test_results], ignore_index=True)
    filename = f"{model_key}_predictions_{timestamp}.csv"
    filepath = os.path.join(predictions_dir, filename)
    all_results.to_csv(filepath, index=False, encoding='utf-8')
    return filepath

# ==================== 统一的交叉验证函数 ====================
def perform_cross_validation(model, X, y, cv_config, need_scaling=True):
    """统一的交叉验证函数（支持KFold和RepeatedKFold）"""
    method = cv_config.get('method', 'kfold')

    if method == 'repeated_kfold':
        cv_splitter = RepeatedKFold(
            n_splits=cv_config['n_splits'],
            n_repeats=cv_config.get('n_repeats', 3),
            random_state=cv_config['random_state']
        )
    else:
        cv_splitter = KFold(
            n_splits=cv_config['n_splits'],
            shuffle=cv_config.get('shuffle', True),
            random_state=cv_config.get('random_state', 2023)
        )

    splits = cv_splitter.split(X)

    cv_scores = []
    fold_predictions = []

    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        X_train_fold = X[train_idx]
        X_val_fold = X[val_idx]
        y_train_fold = y[train_idx]
        y_val_fold = y[val_idx]

        if need_scaling:
            scaler = StandardScaler()
            X_train_fold = scaler.fit_transform(X_train_fold)
            X_val_fold = scaler.transform(X_val_fold)

        try:
            model.fit(X_train_fold, y_train_fold)
            val_pred = model.predict(X_val_fold)
            val_score = r2_score(y_val_fold, val_pred)
            cv_scores.append(val_score)

            fold_predictions.append({
                'fold': fold_idx,
                'val_indices': val_idx,
                'y_true': y_val_fold,
                'y_pred': val_pred,
                'score': val_score
            })
        except Exception as e:
            continue

    if len(cv_scores) == 0:
        return None

    return {
        'mean_score': np.mean(cv_scores),
        'std_score': np.std(cv_scores),
        'all_scores': cv_scores,
        'n_splits': len(cv_scores),
        'fold_predictions': fold_predictions,
        'cv_method': method
    }


# ==================== 阶段1：快速粗筛 ====================
def stage1_quick_filter(X_train, y_train, features_name, feature_tier_map):
    """阶段1：快速粗筛（纯规则）"""
    print("\n" + "="*80)
    print("【阶段1】快速粗筛 - 删除明显垃圾特征（纯规则）")
    print("="*80)
    print(f"输入特征数: {X_train.shape[1]}")

    start_time = time.time()

    keep_mask = np.ones(X_train.shape[1], dtype=bool)
    removal_log = []

    # 步骤1.1：删除常数/近常数特征
    print(f"\n步骤1.1：删除常数/近常数特征")
    print(f"  条件：唯一值比例 < {CONFIG['stage1']['unique_ratio_threshold']}")

    removed_11_features = []
    for i, feat in enumerate(features_name):
        unique_ratio = len(np.unique(X_train[:, i])) / len(X_train[:, i])
        tier = feature_tier_map[feat]

        if tier == 'S':
            continue

        if unique_ratio < CONFIG['stage1']['unique_ratio_threshold']:
            keep_mask[i] = False
            removal_log.append({
                'stage': '1.1',
                'feature': feat,
                'tier': tier,
                'reason': f'近常数特征(唯一值比例={unique_ratio:.4f})'
            })
            removed_11_features.append((feat, tier, unique_ratio))

    removed_count_11 = len(removed_11_features)
    print(f"  删除: {removed_count_11}个")

    # 步骤1.2：删除全零特征
    print(f"\n步骤1.2：删除全零特征")

    removed_12_features = []
    for i, feat in enumerate(features_name):
        if not keep_mask[i]:
            continue

        if np.all(X_train[:, i] == 0):
            keep_mask[i] = False
            tier = feature_tier_map[feat]
            removal_log.append({
                'stage': '1.2',
                'feature': feat,
                'tier': tier,
                'reason': '全零特征'
            })
            removed_12_features.append((feat, tier))

    removed_count_12 = (~keep_mask).sum()
    print(f"  删除: {removed_count_12 - removed_count_11}个")

    # 步骤1.3：删除极度共线特征
    print(f"\n步骤1.3：删除极度共线特征")
    print(f"  阈值：|相关系数| > {CONFIG['stage1']['high_collinearity_threshold']}")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)
    corr_matrix = np.corrcoef(X_scaled, rowvar=False)

    threshold = CONFIG['stage1']['high_collinearity_threshold']
    tier_order = {'S': 3, 'A': 2, 'B': 1}

    removed_13_features = []
    for i in range(len(features_name)):
        if not keep_mask[i]:
            continue
        for j in range(i+1, len(features_name)):
            if not keep_mask[j]:
                continue

            if abs(corr_matrix[i, j]) > threshold:
                feat_i = features_name[i]
                feat_j = features_name[j]
                tier_i = feature_tier_map[feat_i]
                tier_j = feature_tier_map[feat_j]

                if tier_order[tier_i] > tier_order[tier_j]:
                    keep_mask[j] = False
                    reason = f'与{feat_i}极度共线(r={corr_matrix[i,j]:.3f}), {tier_j}级<{tier_i}级'
                    removal_log.append({
                        'stage': '1.3',
                        'feature': feat_j,
                        'tier': tier_j,
                        'reason': reason
                    })
                    removed_13_features.append((feat_j, tier_j, feat_i, corr_matrix[i,j]))

                elif tier_order[tier_i] < tier_order[tier_j]:
                    keep_mask[i] = False
                    reason = f'与{feat_j}极度共线(r={corr_matrix[i,j]:.3f}), {tier_i}级<{tier_j}级'
                    removal_log.append({
                        'stage': '1.3',
                        'feature': feat_i,
                        'tier': tier_i,
                        'reason': reason
                    })
                    removed_13_features.append((feat_i, tier_i, feat_j, corr_matrix[i,j]))
                    break

                else:
                    r_i = abs(np.corrcoef(X_scaled[:, i], y_train)[0, 1])
                    r_j = abs(np.corrcoef(X_scaled[:, j], y_train)[0, 1])

                    if r_i > r_j:
                        keep_mask[j] = False
                        reason = f'与{feat_i}极度共线(r={corr_matrix[i,j]:.3f}), 同级但与目标相关性低'
                        removal_log.append({
                            'stage': '1.3',
                            'feature': feat_j,
                            'tier': tier_j,
                            'reason': reason
                        })
                        removed_13_features.append((feat_j, tier_j, feat_i, corr_matrix[i,j]))
                    else:
                        keep_mask[i] = False
                        reason = f'与{feat_j}极度共线(r={corr_matrix[i,j]:.3f}), 同级但与目标相关性低'
                        removal_log.append({
                            'stage': '1.3',
                            'feature': feat_i,
                            'tier': tier_i,
                            'reason': reason
                        })
                        removed_13_features.append((feat_i, tier_i, feat_j, corr_matrix[i,j]))
                        break

    removed_count_13 = (~keep_mask).sum()
    print(f"  删除: {removed_count_13 - removed_count_12}个")

    keep_indices = np.where(keep_mask)[0].tolist()
    selected_features = [features_name[i] for i in keep_indices]

    elapsed_time = time.time() - start_time

    print(f"\n" + "="*80)
    print("阶段1完成统计:")
    print("="*80)
    print(f"输入特征数: {len(features_name)}")
    print(f"保留特征数: {len(selected_features)}")
    print(f"删除特征数: {len(removal_log)}")
    print(f"\n⏱️  耗时: {elapsed_time:.1f}秒")

    if removal_log:
        report_df = pd.DataFrame(removal_log)
        report_path = os.path.join(CONFIG['output_dir'], 'stage1_quick_filter_report.csv')
        report_df.to_csv(report_path, index=False, encoding='utf-8')
        print(f"\n阶段1删除详细报告已保存: {report_path}")

    return keep_indices, selected_features


# ==================== 阶段2：NMI计算（修复版）====================
def calculate_normalized_mutual_information(X_scaled, y_train, n_bins=10, method='quantile'):
    """
    使用标准化互信息（NMI）计算特征间冗余度
    NMI公式: NMI(X,Y) = 2 * MI(X,Y) / (H(X) + H(Y))
    """
    n_features = X_scaled.shape[1]

    print(f"    正在离散化特征 (方法={method}, bins={n_bins})...")
    X_discrete = np.zeros_like(X_scaled, dtype=int)

    for i in range(n_features):
        feature_data = X_scaled[:, i]

        if method == 'quantile':
            try:
                X_discrete[:, i] = pd.qcut(feature_data, q=n_bins, labels=False, duplicates='drop')
            except:
                X_discrete[:, i] = pd.cut(feature_data, bins=n_bins, labels=False)
        else:
            X_discrete[:, i] = pd.cut(feature_data, bins=n_bins, labels=False)

        if np.isnan(X_discrete[:, i]).any():
            mode_val = pd.Series(X_discrete[:, i]).mode()[0]
            X_discrete[:, i] = np.nan_to_num(X_discrete[:, i], nan=mode_val)

    print(f"    离散化完成，开始计算 {n_features}×{n_features} NMI矩阵...")

    entropies = np.zeros(n_features)
    for i in range(n_features):
        entropies[i] = mutual_info_score(X_discrete[:, i], X_discrete[:, i])

    nmi_matrix = np.zeros((n_features, n_features))

    for i in range(n_features):
        nmi_matrix[i, i] = 1.0

        for j in range(i+1, n_features):
            mi_ij = mutual_info_score(X_discrete[:, i], X_discrete[:, j])

            H_i = entropies[i]
            H_j = entropies[j]

            if (H_i + H_j) > 0:
                nmi = 2.0 * mi_ij / (H_i + H_j)
            else:
                nmi = 0.0

            nmi_matrix[i, j] = nmi
            nmi_matrix[j, i] = nmi

        if (i+1) % 20 == 0:
            print(f"      进度: {i+1}/{n_features}")

    print(f"    ✅ NMI矩阵计算完成（使用标准化互信息）")

    return nmi_matrix


def stage2_integrated_selection(X_train, y_train, features_name, feature_tier_map):
    """阶段2：综合筛选"""
    print("\n" + "="*80)
    print("【阶段2】综合筛选 - 评分+去冗余一步完成（模型不可知）")
    print("="*80)
    print(f"输入特征数: {X_train.shape[1]}")
    print(f"✅ MI计算方法: 离散化({CONFIG['stage2']['mi_discretization']['n_bins']}bins) + NMI归一化")

    start_time = time.time()
    target_n = CONFIG['stage2_target']

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)

    print("\n" + "-"*80)
    print("步骤2.1：计算Relevance（模型不可知的多维度评分）")
    print("-"*80)

    n_features = X_train.shape[1]

    print("\n  指标1：计算Pearson相关（45%权重）...")
    pearson_scores = []
    for i in range(n_features):
        r, _ = pearsonr(X_scaled[:, i], y_train)
        pearson_scores.append(abs(r))
    pearson_scores = np.array(pearson_scores)

    print("  指标2：计算互信息MI（35%权重）...")
    mi_scores = mutual_info_regression(X_scaled, y_train, random_state=CONFIG['random_state'])

    print("  指标3：计算目标条件方差（20%权重）...")
    n_bins = 5
    y_percentiles = np.percentile(y_train, np.linspace(0, 100, n_bins + 1))

    discriminative_scores = []
    for i in range(n_features):
        group_vars = []
        for j in range(n_bins):
            mask = (y_train >= y_percentiles[j]) & (y_train < y_percentiles[j+1])
            if np.sum(mask) > 1:
                group_vars.append(np.var(X_scaled[mask, i]))

        if len(group_vars) > 0:
            discriminative_scores.append(np.std(group_vars))
        else:
            discriminative_scores.append(0)

    discriminative_scores = np.array(discriminative_scores)

    dcor_scores = None
    if DCOR_AVAILABLE:
        print("  可选指标：计算距离相关dCor（20%权重）...")
        dcor_scores = []
        for i in range(n_features):
            dcor = distance_correlation(X_scaled[:, i], y_train)
            dcor_scores.append(dcor)
        dcor_scores = np.array(dcor_scores)

    print("\n  Rank归一化...")
    pearson_ranks = rankdata(pearson_scores) / n_features
    mi_ranks = rankdata(mi_scores) / n_features
    discriminative_ranks = rankdata(discriminative_scores) / n_features

    if dcor_scores is not None:
        dcor_ranks = rankdata(dcor_scores) / n_features

    if dcor_scores is not None:
        weights = CONFIG['stage2']['weights_with_dcor']
        base_scores = (weights['pearson'] * pearson_ranks +
                      weights['mi'] * mi_ranks +
                      weights['dcor'] * dcor_ranks +
                      weights['discriminative'] * discriminative_ranks)
    else:
        weights = CONFIG['stage2']['weights']
        base_scores = (weights['pearson'] * pearson_ranks +
                      weights['mi'] * mi_ranks +
                      weights['discriminative'] * discriminative_ranks)

    print("\n  应用等级加权...")
    tier_multipliers = CONFIG['stage2']['tier_multipliers']

    relevance = base_scores.copy()
    for i, feat in enumerate(features_name):
        tier = feature_tier_map[feat]
        multiplier = tier_multipliers[tier]
        relevance[i] = base_scores[i] * multiplier

    print("\n" + "-"*80)
    print("步骤2.2：mRMR迭代选择（用NMI度量冗余）")
    print("-"*80)
    print(f"  目标选择: {target_n}个特征")

    print("\n  ✅ 使用NMI（标准化互信息）计算特征间冗余度...")
    mi_config = CONFIG['stage2']['mi_discretization']

    nmi_matrix = calculate_normalized_mutual_information(
        X_scaled,
        y_train,
        n_bins=mi_config['n_bins'],
        method=mi_config['method']
    )

    print("\n  开始mRMR迭代选择...")
    selected_indices = []
    remaining_indices = list(range(n_features))

    first_idx = np.argmax(relevance)
    selected_indices.append(first_idx)
    remaining_indices.remove(first_idx)

    print(f"    第1/{target_n}个: {features_name[first_idx]} (relevance={relevance[first_idx]:.4f})")

    for step in range(1, target_n):
        best_score = -np.inf
        best_idx = None

        for idx in remaining_indices:
            redundancy = np.mean([nmi_matrix[idx, s] for s in selected_indices])
            mrmr_score = relevance[idx] - redundancy

            if mrmr_score > best_score:
                best_score = mrmr_score
                best_idx = idx

        selected_indices.append(best_idx)
        remaining_indices.remove(best_idx)

        if (step + 1) % 10 == 0 or (step + 1) == target_n:
            redundancy_val = np.mean([nmi_matrix[best_idx, s] for s in selected_indices[:-1]])
            print(f"    第{step+1}/{target_n}个: {features_name[best_idx]} "
                  f"(relevance={relevance[best_idx]:.4f}, redundancy={redundancy_val:.4f})")

    selected_features = [features_name[i] for i in selected_indices]

    elapsed_time = time.time() - start_time

    print(f"\n" + "="*80)
    print("阶段2完成统计:")
    print("="*80)
    print(f"输入特征数: {len(features_name)}")
    print(f"输出特征数: {len(selected_features)}")
    print(f"✅ MI计算: 离散化({mi_config['n_bins']}bins) + NMI归一化")
    print(f"\n⏱️  耗时: {elapsed_time:.1f}秒")

    return selected_indices, selected_features


# ==================== 阶段4：树模型组特征选择 ====================
def stage4_tree_model_selection(X_train, y_train, features_60, feature_tier_map):
    """阶段4：树模型组特征选择"""
    print("\n" + "="*80)
    print("【阶段4】树模型组特征选择")
    print("="*80)
    print(f"输入特征数: {X_train.shape[1]}")
    print(f"目标选择: {CONFIG['stage4_tree_target']}个特征")

    start_time = time.time()

    print("\n  方法1：训练RandomForest获取特征重要性（50%权重）...")
    rf_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_split=5,
        random_state=CONFIG['random_state'],
        n_jobs=CONFIG['n_jobs']
    )
    rf_model.fit(X_train, y_train)
    rf_importance = rf_model.feature_importances_
    print(f"    ✓ RF训练完成，Top5特征: {', '.join([features_60[i] for i in np.argsort(rf_importance)[::-1][:5]])}")

    print("\n  方法2：训练XGBoost获取特征重要性（40%权重）...")
    xgb_model = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        random_state=CONFIG['random_state'],
        n_jobs=CONFIG['n_jobs'],
        verbosity=0
    )
    xgb_model.fit(X_train, y_train)
    xgb_importance = xgb_model.feature_importances_
    print(f"    ✓ XGB训练完成，Top5特征: {', '.join([features_60[i] for i in np.argsort(xgb_importance)[::-1][:5]])}")

    print("\n  方法3：训练ExtraTrees获取特征重要性（10%权重）...")
    et_model = ExtraTreesRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        random_state=CONFIG['random_state'],
        n_jobs=CONFIG['n_jobs']
    )
    et_model.fit(X_train, y_train)
    et_importance = et_model.feature_importances_
    print(f"    ✓ ET训练完成，Top5特征: {', '.join([features_60[i] for i in np.argsort(et_importance)[::-1][:5]])}")

    print("\n  归一化各模型的重要性得分...")
    rf_norm = (rf_importance - rf_importance.min()) / (rf_importance.max() - rf_importance.min() + 1e-10)
    xgb_norm = (xgb_importance - xgb_importance.min()) / (xgb_importance.max() - xgb_importance.min() + 1e-10)
    et_norm = (et_importance - et_importance.min()) / (et_importance.max() - et_importance.min() + 1e-10)

    base_score_tree = 0.5 * rf_norm + 0.4 * xgb_norm + 0.1 * et_norm

    print("\n  应用特征等级权重...")
    weighted_tree = []
    for i, feat in enumerate(features_60):
        tier = feature_tier_map[feat]
        weight = CONFIG['feature_tiers'][tier]['weight']
        weighted_score = base_score_tree[i] * weight
        weighted_tree.append({
            'index': i,
            'feature': feat,
            'tier': tier,
            'base_score': base_score_tree[i],
            'weighted_score': weighted_score,
            'rf_importance': rf_importance[i],
            'xgb_importance': xgb_importance[i],
            'et_importance': et_importance[i]
        })

    weighted_tree.sort(key=lambda x: x['weighted_score'], reverse=True)

    tree_group_top = weighted_tree[:CONFIG['stage4_tree_target']]
    tree_group_indices = [item['index'] for item in tree_group_top]
    tree_group_features = [item['feature'] for item in tree_group_top]

    elapsed_time = time.time() - start_time

    print(f"\n" + "="*80)
    print("阶段4完成统计（树模型组）:")
    print("="*80)
    print(f"输入特征数: {len(features_60)}")
    print(f"输出特征数: {len(tree_group_features)}")
    print(f"\n✓ Top{CONFIG['stage4_tree_target']}特征列表:")
    for rank, item in enumerate(tree_group_top, 1):
        print(f"  {rank:>2}. [{item['tier']}] {item['feature']:<20} | "
              f"综合={item['weighted_score']:.4f} | "
              f"RF={item['rf_importance']:.4f} | "
              f"XGB={item['xgb_importance']:.4f} | "
              f"ET={item['et_importance']:.4f}")

    print(f"\n⏱️  耗时: {elapsed_time:.1f}秒")

    report_df = pd.DataFrame(weighted_tree)
    report_path = os.path.join(CONFIG['output_dir'], 'stage4_tree_group_feature_selection.csv')
    report_df.to_csv(report_path, index=False, encoding='utf-8')
    print(f"\n树模型组特征选择详细报告已保存: {report_path}")

    return {
        'indices': tree_group_indices,
        'features': tree_group_features,
        'detailed_scores': tree_group_top
    }


# ==================== 阶段5：树模型组穷举搜索 ====================
def evaluate_combination_tree_group(feature_indices, X_subset, y_train, random_state):
    """阶段5专用：用简单集成评估树模型组特征组合"""
    np.random.seed(random_state)

    X_comb = X_subset[:, list(feature_indices)]

    models = [
        ('RF', RandomForestRegressor(
            n_estimators=100,
            max_depth=8,
            min_samples_split=5,
            random_state=random_state,
            n_jobs=1
        )),
        ('XGB', XGBRegressor(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.05,
            random_state=random_state,
            n_jobs=1,
            verbosity=0
        ))
    ]

    need_scaling = False
    cv_config = CONFIG['cv_strategy']['stage5']
    all_model_scores = []

    for model_name, model in models:
        try:
            cv_result = perform_cross_validation(
                model, X_comb, y_train, cv_config, need_scaling=need_scaling
            )

            if cv_result is not None:
                all_model_scores.append(cv_result['mean_score'])
        except Exception as e:
            continue

    if len(all_model_scores) > 0:
        ensemble_score = np.mean(all_model_scores)
    else:
        ensemble_score = -np.inf

    return {
        'feature_indices': list(feature_indices),
        'cv_r2': ensemble_score
    }


def stage5_tree_group_exhaustive_search(X_train_subset, y_train, group_features):
    """阶段5：树模型组K=4-7穷举搜索"""
    print("\n" + "="*80)
    print("【阶段5】树模型组K=4-7穷举搜索")
    print("="*80)
    print(f"特征池: {len(group_features)}个特征")
    print(f"K值范围: {CONFIG['stage5']['k_range']}")
    print(f"并行进程数: {CONFIG['stage5']['n_jobs_parallel']}")
    print(f"✅ CV策略: {CONFIG['cv_strategy']['stage5']['method']} "
          f"({CONFIG['cv_strategy']['stage5']['n_splits']}折)")

    start_time = time.time()

    results_all = []
    n_jobs_parallel = CONFIG['stage5']['n_jobs_parallel']

    for k in CONFIG['stage5']['k_range']:
        from math import comb
        n_combinations = comb(len(group_features), k)
        print(f"\n  K={k}: 共{n_combinations}个组合", end="", flush=True)

        k_start_time = time.time()

        all_combinations = list(itertools.combinations(range(len(group_features)), k))

        k_results_raw = Parallel(n_jobs=n_jobs_parallel, backend='loky', verbose=0)(
            delayed(evaluate_combination_tree_group)(
                comb, X_train_subset, y_train, CONFIG['random_state']
            ) for comb in all_combinations
        )

        k_results = []
        for result in k_results_raw:
            selected_feats = [group_features[i] for i in result['feature_indices']]
            k_results.append({
                'k': k,
                'feature_indices': result['feature_indices'],
                'features': selected_feats,
                'cv_r2': result['cv_r2']
            })

        k_results.sort(key=lambda x: x['cv_r2'], reverse=True)

        top_n = min(CONFIG['stage5']['top_n_per_k'], len(k_results))
        k_top_results = k_results[:top_n]

        results_all.extend(k_top_results)

        k_elapsed = time.time() - k_start_time
        print(f" → Top{top_n}组合, 最佳CV R²={k_top_results[0]['cv_r2']:.4f} "
              f"(耗时{k_elapsed:.1f}秒)", flush=True)

    elapsed_time = time.time() - start_time

    print(f"\n" + "="*80)
    print("阶段5完成统计（树模型组）:")
    print("="*80)
    print(f"总评估组合数: 共{len(results_all)}个最优组合")
    print(f"⏱️  总耗时: {elapsed_time/60:.1f}分钟")

    results_df = pd.DataFrame([
        {
            'K值': r['k'],
            'CV_R2': r['cv_r2'],
            '特征数量': len(r['features']),
            '特征列表': ', '.join(r['features'])
        } for r in results_all
    ])
    results_path = os.path.join(CONFIG['output_dir'], 'stage5_tree_group_combinations.csv')
    results_df.to_csv(results_path, index=False, encoding='utf-8')
    print(f"\n树模型组穷举结果已保存: {results_path}")

    return results_all


# ==================== 阶段6：树模型组最终训练 ====================
# ✅ v2.2修改：新增训练集MAE和RMSE的存储
def evaluate_single_tree_model_stage6(args):
    """阶段6：评估单个树模型"""
    (X_train_comb, X_test_comb, y_train, y_test,
     train_indices, test_indices, model_name, base_model,
     k, comb_idx, feature_names,
     cv_config, timestamp) = args

    model_key = f"Tree_{model_name}_K{k}_Comb{comb_idx+1}"

    repeat_seeds = [CONFIG['base_random_state'] + i
                   for i in range(CONFIG['n_repeat_seeds'])]

    cv_r2_list = []
    train_r2_list = []
    train_mae_list = []      # ✅ v2.2新增
    train_rmse_list = []     # ✅ v2.2新增
    test_r2_list = []
    test_mae_list = []
    test_rmse_list = []      # ✅ v2.2新增
    gap_list = []
    convergence_warnings = []

    need_scaling = False

    for seed_idx, seed in enumerate(repeat_seeds):
        model = clone(base_model)

        valid_params = model.get_params().keys()
        if 'random_state' in valid_params:
            try:
                model.set_params(random_state=seed)
            except (ValueError, TypeError):
                pass

        X_train_current = X_train_comb
        y_train_current = y_train

        cv_result = perform_cross_validation(
            model, X_train_current, y_train_current, cv_config,
            need_scaling=need_scaling
        )

        if cv_result is None:
            continue

        cv_r2_list.append(cv_result['mean_score'])

        X_train_scaled = X_train_current
        X_test_scaled = X_test_comb

        try:
            model.fit(X_train_scaled, y_train_current)

            pred_train = model.predict(X_train_scaled)
            pred_test = model.predict(X_test_scaled)

            train_metrics = calculate_comprehensive_metrics(y_train, pred_train)
            test_metrics = calculate_comprehensive_metrics(y_test, pred_test)

            train_r2_list.append(train_metrics['r2'])
            train_mae_list.append(train_metrics['mae'])    # ✅ v2.2新增
            train_rmse_list.append(train_metrics['rmse'])  # ✅ v2.2新增
            test_r2_list.append(test_metrics['r2'])
            test_mae_list.append(test_metrics['mae'])
            test_rmse_list.append(test_metrics['rmse'])    # ✅ v2.2新增
            gap_list.append(train_metrics['r2'] - test_metrics['r2'])

            if seed_idx == 0:
                first_model = model
                first_pred_train = pred_train
                first_pred_test = pred_test

        except Exception as e:
            continue

    if len(cv_r2_list) < CONFIG['n_repeat_seeds'] // 2:
        return None

    mean_cv_r2 = np.mean(cv_r2_list)
    std_cv_r2 = np.std(cv_r2_list)
    mean_train_r2 = np.mean(train_r2_list)
    std_train_r2 = np.std(train_r2_list)
    mean_train_mae = np.mean(train_mae_list)    # ✅ v2.2新增
    std_train_mae = np.std(train_mae_list)      # ✅ v2.2新增
    mean_train_rmse = np.mean(train_rmse_list)  # ✅ v2.2新增
    std_train_rmse = np.std(train_rmse_list)    # ✅ v2.2新增
    mean_test_r2 = np.mean(test_r2_list)
    std_test_r2 = np.std(test_r2_list)
    mean_gap = np.mean(gap_list)
    std_gap = np.std(gap_list)
    mean_test_mae = np.mean(test_mae_list)
    std_test_mae = np.std(test_mae_list)
    mean_test_rmse = np.mean(test_rmse_list)    # ✅ v2.2新增
    std_test_rmse = np.std(test_rmse_list)      # ✅ v2.2新增

    is_qualified = (mean_test_r2 >= CONFIG['min_r2_threshold'] and
                   mean_test_r2 <= CONFIG['max_r2_threshold'] and
                   mean_gap <= CONFIG['max_generalization_gap'])

    result = {
        'model_key': model_key,
        'group': 'Tree',
        'model': model_name,
        'k': k,
        'features': feature_names,
        'mean_cv_r2': mean_cv_r2,
        'std_cv_r2': std_cv_r2,
        'mean_train_r2': mean_train_r2,
        'std_train_r2': std_train_r2,
        'mean_train_mae': mean_train_mae,      # ✅ v2.2新增
        'std_train_mae': std_train_mae,        # ✅ v2.2新增
        'mean_train_rmse': mean_train_rmse,    # ✅ v2.2新增
        'std_train_rmse': std_train_rmse,      # ✅ v2.2新增
        'mean_test_r2': mean_test_r2,
        'std_test_r2': std_test_r2,
        'mean_gap': mean_gap,
        'std_gap': std_gap,
        'mean_test_mae': mean_test_mae,
        'std_test_mae': std_test_mae,
        'mean_test_rmse': mean_test_rmse,      # ✅ v2.2新增
        'std_test_rmse': std_test_rmse,        # ✅ v2.2新增
        'n_evaluations': len(cv_r2_list),
        'convergence_warnings': convergence_warnings,
        'is_qualified': is_qualified,
        'randomization_strategy': 'random_state'
    }

    if is_qualified:
        pred_file = save_detailed_predictions(
            model_key, y_train, first_pred_train,
            y_test, first_pred_test,
            train_indices, test_indices, timestamp
        )
        result['model_instance'] = first_model
        result['scaler'] = None
        result['predictions_file'] = pred_file

    return result


def stage6_tree_group_final_training(X_train_stage2, X_test_stage2, y_train, y_test,
                                     train_indices, test_indices, tree_combinations,
                                     tree_group_indices):
    """阶段6：树模型组最终训练与评估"""
    print("\n" + "="*80)
    print("【阶段6】树模型组最终训练与评估（✅ 优化版）")
    print("="*80)

    timestamp = time.strftime("%Y%m%d_%H%M%S")

    cv_config = CONFIG['cv_strategy']['stage6']
    print(f"✅ CV策略: {cv_config['method']} ({cv_config['n_splits']}折)")
    print(f"🚀 并行线程数: {CONFIG['stage6_n_jobs']}个")
    print(f"🔥 随机化策略: random_state（{CONFIG['n_repeat_seeds']}个种子）")

    model_library = {
        'RandomForest': RandomForestRegressor(
            n_estimators=150,
            max_depth=6,
            min_samples_split=8,
            min_samples_leaf=4,
            max_features='sqrt',
            random_state=CONFIG['random_state'],
            n_jobs=1
        ),
        'XGBoost': XGBRegressor(
            n_estimators=150,
            max_depth=4,
            learning_rate=0.08,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=CONFIG['random_state'],
            n_jobs=1,
            verbosity=0
        ),
        'GradientBoosting': GradientBoostingRegressor(
            n_estimators=120,
            max_depth=3,
            learning_rate=0.08,
            subsample=0.8,
            min_samples_split=8,
            min_samples_leaf=4,
            random_state=CONFIG['random_state']
        ),
        'ExtraTrees': ExtraTreesRegressor(
            n_estimators=200,
            max_depth=10,
            min_samples_split=5,
            random_state=CONFIG['random_state'],
            n_jobs=1
        ),
        'HistGradientBoosting': HistGradientBoostingRegressor(
            max_iter=150,
            max_depth=8,
            learning_rate=0.05,
            random_state=CONFIG['random_state']
        )
    }

    print("\n构建评估任务列表...")
    all_tasks = []

    for comb_idx, comb in enumerate(tree_combinations):
        k = comb['k']
        feature_names = comb['features']
        feature_indices_in_subset = comb['feature_indices']

        feature_indices_in_stage2 = [tree_group_indices[i] for i in feature_indices_in_subset]

        X_train_comb = X_train_stage2[:, feature_indices_in_stage2]
        X_test_comb = X_test_stage2[:, feature_indices_in_stage2]

        for model_name, base_model in model_library.items():
            task_args = (
                X_train_comb, X_test_comb, y_train, y_test,
                train_indices, test_indices, model_name, base_model,
                k, comb_idx, feature_names,
                cv_config, timestamp
            )
            all_tasks.append(task_args)

    total_tasks = len(all_tasks)
    print(f"✅ 共构建 {total_tasks} 个评估任务")
    print(f"   = {len(tree_combinations)}个特征组合 × {len(model_library)}个模型")
    print(f"🚀 开始{CONFIG['stage6_n_jobs']}线程并行评估...\n")

    start_time = time.time()

    results_raw = Parallel(n_jobs=CONFIG['stage6_n_jobs'], backend='loky', verbose=10)(
        delayed(evaluate_single_tree_model_stage6)(task) for task in all_tasks
    )

    elapsed_time = time.time() - start_time

    all_results = []
    all_qualified_models = []

    for result in results_raw:
        if result is None:
            continue

        # ✅ v2.2修改：all_results中新增训练集MAE和RMSE字段
        all_results.append({
            'model_key': result['model_key'],
            'group': result['group'],
            'model': result['model'],
            'k': result['k'],
            'mean_cv_r2': result['mean_cv_r2'],
            'std_cv_r2': result['std_cv_r2'],
            'mean_train_r2': result['mean_train_r2'],
            'std_train_r2': result['std_train_r2'],
            'mean_train_mae': result['mean_train_mae'],      # ✅ v2.2新增
            'std_train_mae': result['std_train_mae'],        # ✅ v2.2新增
            'mean_train_rmse': result['mean_train_rmse'],    # ✅ v2.2新增
            'std_train_rmse': result['std_train_rmse'],      # ✅ v2.2新增
            'mean_test_r2': result['mean_test_r2'],
            'std_test_r2': result['std_test_r2'],
            'mean_gap': result['mean_gap'],
            'std_gap': result['std_gap'],
            'mean_test_mae': result['mean_test_mae'],
            'std_test_mae': result['std_test_mae'],
            'mean_test_rmse': result['mean_test_rmse'],      # ✅ v2.2新增
            'std_test_rmse': result['std_test_rmse'],        # ✅ v2.2新增
            'n_evaluations': result['n_evaluations'],
            'convergence_warnings': len(result['convergence_warnings']),
            'randomization_strategy': result['randomization_strategy']
        })

        if result['is_qualified']:
            all_qualified_models.append(result)

    print(f"\n" + "="*80)
    print("🚀 阶段6并行评估完成统计（✅ 优化版）:")
    print("="*80)
    print(f"总评估任务数: {total_tasks}")
    print(f"有效评估数: {len(all_results)}")
    print(f"合格模型数: {len(all_qualified_models)}")
    print(f"⏱️  总耗时: {elapsed_time/60:.1f}分钟")
    print(f"⚡ 平均每任务: {elapsed_time/total_tasks:.1f}秒")

    model_counter = Counter([r['model'] for r in all_qualified_models])
    print(f"\n📊 按模型类型统计合格模型:")
    for model_name in model_library.keys():
        count = model_counter.get(model_name, 0)
        print(f"  {model_name:25s}: {count:3d}个")

    print("="*80)

    return all_results, all_qualified_models, timestamp


# ==================== 阶段7：Top100评估与Top30筛选 ====================
def sort_models_by_test_r2_with_stability(qualified_models):
    """按测试集R²均值排序（含稳定性惩罚）"""
    print("\n" + "="*80)
    print("【模型排序】按测试集R²均值排序（含稳定性惩罚）")
    print("="*80)

    penalty = CONFIG['stability_penalty']
    print(f"\n排序策略:")
    print(f"  排序得分 = Mean_Test_R² - {penalty} × Std_Test_R²")

    for model in qualified_models:
        model['sort_score'] = (model['mean_test_r2'] -
                               penalty * model['std_test_r2'])

    qualified_models_sorted = sorted(qualified_models,
                                     key=lambda x: x['sort_score'],
                                     reverse=True)

    print(f"\n✅ 排序完成! Top30模型预览:")
    print(f"{'='*80}")

    for rank, model in enumerate(qualified_models_sorted[:30], 1):
        convergence_str = ""
        if model.get('convergence_warnings') and len(model['convergence_warnings']) > 0:
            convergence_str = " ⚠️"

        print(f"{rank:>2}. {model['model_key']:<50} | "
              f"排序={model['sort_score']:.4f} | "
              f"CV R²={model['mean_cv_r2']:.4f}±{model['std_cv_r2']:.4f} | "
              f"Test R²={model['mean_test_r2']:.4f}±{model['std_test_r2']:.4f}{convergence_str}")

    return qualified_models_sorted


# ✅ v2.2修改：新增训练集MAE和RMSE的计算、存储、打印
def evaluate_on_true_test_set(model_info, X_train_full, y_train_full,
                               X_test_true, y_test_true, features_60):
    """在真实独立测试集上评估单个树模型"""
    model_key = model_info['model_key']
    model_features = model_info['features']

    feature_indices = [features_60.index(feat) for feat in model_features]
    X_train_model = X_train_full[:, feature_indices]
    X_test_model = X_test_true[:, feature_indices]

    need_scaling = False
    repeat_seeds = [CONFIG['base_random_state'] + i for i in range(CONFIG['n_repeat_seeds'])]

    train_r2_list = []
    train_mae_list = []    # ✅ v2.2新增
    train_rmse_list = []   # ✅ v2.2新增
    test_r2_list = []
    test_mae_list = []
    test_rmse_list = []

    model_name = model_info['model']

    if 'RandomForest' in model_name:
        base_model = RandomForestRegressor(
            n_estimators=150, max_depth=6, min_samples_split=8,
            min_samples_leaf=4, max_features='sqrt',
            random_state=CONFIG['random_state'], n_jobs=1
        )
    elif 'XGBoost' in model_name:
        base_model = XGBRegressor(
            n_estimators=150, max_depth=4, learning_rate=0.08,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0,
            random_state=CONFIG['random_state'], n_jobs=1, verbosity=0
        )
    elif 'GradientBoosting' in model_name:
        base_model = GradientBoostingRegressor(
            n_estimators=120, max_depth=3, learning_rate=0.08,
            subsample=0.8, min_samples_split=8, min_samples_leaf=4,
            random_state=CONFIG['random_state']
        )
    elif 'ExtraTrees' in model_name:
        base_model = ExtraTreesRegressor(
            n_estimators=200, max_depth=10, min_samples_split=5,
            random_state=CONFIG['random_state'], n_jobs=1
        )
    elif 'HistGradientBoosting' in model_name:
        base_model = HistGradientBoostingRegressor(
            max_iter=150, max_depth=8, learning_rate=0.05,
            random_state=CONFIG['random_state']
        )
    else:
        base_model = model_info['model_instance']

    for seed in repeat_seeds:
        model = clone(base_model)

        valid_params = model.get_params().keys()
        if 'random_state' in valid_params:
            try:
                model.set_params(random_state=seed)
            except (ValueError, TypeError):
                pass

        try:
            model.fit(X_train_model, y_train_full)

            pred_train = model.predict(X_train_model)
            pred_test = model.predict(X_test_model)

            train_metrics = calculate_comprehensive_metrics(y_train_full, pred_train)
            test_metrics = calculate_comprehensive_metrics(y_test_true, pred_test)

            train_r2_list.append(train_metrics['r2'])
            train_mae_list.append(train_metrics['mae'])    # ✅ v2.2新增
            train_rmse_list.append(train_metrics['rmse'])  # ✅ v2.2新增
            test_r2_list.append(test_metrics['r2'])
            test_mae_list.append(test_metrics['mae'])
            test_rmse_list.append(test_metrics['rmse'])

        except Exception as e:
            continue

    if len(train_r2_list) == 0:
        return None

    result = {
        'model_key': model_key,
        'mean_train_r2_true': np.mean(train_r2_list),
        'std_train_r2_true': np.std(train_r2_list),
        'mean_train_mae_true': np.mean(train_mae_list),    # ✅ v2.2新增
        'std_train_mae_true': np.std(train_mae_list),      # ✅ v2.2新增
        'mean_train_rmse_true': np.mean(train_rmse_list),  # ✅ v2.2新增
        'std_train_rmse_true': np.std(train_rmse_list),    # ✅ v2.2新增
        'mean_test_r2_true': np.mean(test_r2_list),
        'std_test_r2_true': np.std(test_r2_list),
        'mean_test_mae_true': np.mean(test_mae_list),
        'std_test_mae_true': np.std(test_mae_list),
        'mean_test_rmse_true': np.mean(test_rmse_list),
        'std_test_rmse_true': np.std(test_rmse_list),
        'true_gap': np.mean(train_r2_list) - np.mean(test_r2_list),
        'n_seeds': len(train_r2_list)
    }

    return result


def stage7_evaluate_top_models_on_true_test(top_models, X_train_stage2, X_test_stage2,
                                            y_train, y_test, features_60, timestamp):
    """阶段7：在真实独立测试集上评估Top100模型，并选出Top30"""
    print("\n" + "="*80)
    print("【阶段7】Top100模型在独立测试集上评估（树模型组）")
    print("="*80)

    top100_models = top_models[:min(100, len(top_models))]
    print(f"选择Top{len(top100_models)}模型进行真实测试集评估...")

    print(f"\n解封测试集:")
    print(f"  训练集: {X_train_stage2.shape[0]} 样本")
    print(f"  测试集: {X_test_stage2.shape[0]} 样本（真实独立）")

    print(f"\n开始评估Top{len(top100_models)}模型（使用{CONFIG['n_repeat_seeds']}个随机种子）...")

    start_time = time.time()

    true_test_results = Parallel(n_jobs=CONFIG['stage6_n_jobs'], backend='loky', verbose=10)(
        delayed(evaluate_on_true_test_set)(
            model_info, X_train_stage2, y_train, X_test_stage2, y_test, features_60
        ) for model_info in top100_models
    )

    elapsed_time = time.time() - start_time

    top100_with_true_test = []

    for i, model_info in enumerate(top100_models):
        true_result = true_test_results[i]

        if true_result is None:
            continue

        # ✅ v2.2修改：combined中新增训练集MAE和RMSE字段
        combined = {
            'rank_training': i + 1,
            'model_key': model_info['model_key'],
            'group': model_info['group'],
            'model': model_info['model'],
            'k': model_info['k'],
            'features': model_info['features'],
            'randomization_strategy': model_info['randomization_strategy'],

            # 训练集留出验证的性能
            'mean_cv_r2_training': model_info['mean_cv_r2'],
            'std_cv_r2_training': model_info['std_cv_r2'],
            'mean_train_r2_training': model_info['mean_train_r2'],
            'std_train_r2_training': model_info['std_train_r2'],
            'mean_train_mae_training': model_info['mean_train_mae'],    # ✅ v2.2新增
            'std_train_mae_training': model_info['std_train_mae'],      # ✅ v2.2新增
            'mean_train_rmse_training': model_info['mean_train_rmse'],  # ✅ v2.2新增
            'std_train_rmse_training': model_info['std_train_rmse'],    # ✅ v2.2新增
            'mean_test_r2_training': model_info['mean_test_r2'],
            'std_test_r2_training': model_info['std_test_r2'],
            'mean_gap_training': model_info['mean_gap'],

            # 真实测试集的性能
            'mean_train_r2_true': true_result['mean_train_r2_true'],
            'std_train_r2_true': true_result['std_train_r2_true'],
            'mean_train_mae_true': true_result['mean_train_mae_true'],    # ✅ v2.2新增
            'std_train_mae_true': true_result['std_train_mae_true'],      # ✅ v2.2新增
            'mean_train_rmse_true': true_result['mean_train_rmse_true'],  # ✅ v2.2新增
            'std_train_rmse_true': true_result['std_train_rmse_true'],    # ✅ v2.2新增
            'mean_test_r2_true': true_result['mean_test_r2_true'],
            'std_test_r2_true': true_result['std_test_r2_true'],
            'mean_test_mae_true': true_result['mean_test_mae_true'],
            'std_test_mae_true': true_result['std_test_mae_true'],
            'mean_test_rmse_true': true_result['mean_test_rmse_true'],
            'std_test_rmse_true': true_result['std_test_rmse_true'],
            'true_gap': true_result['true_gap'],

            'model_instance': model_info['model_instance'],
            'scaler': model_info.get('scaler', None)
        }

        top100_with_true_test.append(combined)

    print(f"\n✅ Top{len(top100_with_true_test)}真实测试集评估完成")
    print(f"⏱️  耗时: {elapsed_time/60:.1f}分钟")

    # ✅ v2.2修改：Top100 Excel中新增训练集MAE和RMSE列
    top100_data = []
    for model in top100_with_true_test:
        top100_data.append({
            '训练排名': model['rank_training'],
            '模型标识': model['model_key'],
            '组别': model['group'],
            '模型类型': model['model'],
            'K值': model['k'],
            '随机化策略': 'Random State',

            # 训练集性能（留出验证）
            'CV_R²_Training': f"{model['mean_cv_r2_training']:.4f}±{model['std_cv_r2_training']:.4f}",
            'Train_R²_Training': f"{model['mean_train_r2_training']:.4f}±{model['std_train_r2_training']:.4f}",
            'Train_MAE_Training': f"{model['mean_train_mae_training']:.4f}±{model['std_train_mae_training']:.4f}",   # ✅ v2.2新增
            'Train_RMSE_Training': f"{model['mean_train_rmse_training']:.4f}±{model['std_train_rmse_training']:.4f}", # ✅ v2.2新增
            'Test_R²_Training': f"{model['mean_test_r2_training']:.4f}±{model['std_test_r2_training']:.4f}",
            'Gap_Training': f"{model['mean_gap_training']:.4f}",

            # 真实测试性能
            'Train_R²_True': f"{model['mean_train_r2_true']:.4f}±{model['std_train_r2_true']:.4f}",
            'Train_MAE_True': f"{model['mean_train_mae_true']:.4f}±{model['std_train_mae_true']:.4f}",     # ✅ v2.2新增
            'Train_RMSE_True': f"{model['mean_train_rmse_true']:.4f}±{model['std_train_rmse_true']:.4f}",   # ✅ v2.2新增
            'Test_R²_True': f"{model['mean_test_r2_true']:.4f}±{model['std_test_r2_true']:.4f}",
            'Test_MAE_True': f"{model['mean_test_mae_true']:.4f}±{model['std_test_mae_true']:.4f}",
            'Test_RMSE_True': f"{model['mean_test_rmse_true']:.4f}±{model['std_test_rmse_true']:.4f}",
            'Gap_True': f"{model['true_gap']:.4f}",

            '特征数量': len(model['features']),
            '特征列表': ', '.join(model['features'])
        })

    top100_df = pd.DataFrame(top100_data)
    top100_excel = os.path.join(top100_dir, f'top100_true_test_performance_{timestamp}.xlsx')

    with pd.ExcelWriter(top100_excel, engine='openpyxl') as writer:
        # Sheet1: 核心性能对比（✅ v2.2新增训练集MAE和RMSE）
        top100_df[['训练排名', '模型标识', '组别', '模型类型', 'K值',
                   'Train_R²_Training', 'Train_MAE_Training', 'Train_RMSE_Training',
                   'Test_R²_Training',
                   'Train_R²_True', 'Train_MAE_True', 'Train_RMSE_True',
                   'Test_R²_True',
                   'Gap_Training', 'Gap_True']].to_excel(
            writer, sheet_name='核心性能对比', index=False
        )
        # Sheet2: 完整信息
        top100_df.to_excel(writer, sheet_name='完整信息', index=False)

    print(f"\n✅ Top{len(top100_with_true_test)}详细结果已保存: {top100_excel}")

    print(f"\n" + "="*80)
    print("从Top100中选择Top30（优化的筛选+排序策略）")
    print("="*80)

    print(f"\n✅ 综合得分公式:")
    print(f"  方法：筛选 + 排序")
    print(f"    - 先筛选：Gap < 0.20 且 Std < 0.10")
    print(f"    - 再排序：按 Test_R²_True 降序")

    qualified_for_top30 = []
    for model in top100_with_true_test:
        if (abs(model['true_gap']) < 0.20 and
            model['std_test_r2_true'] < 0.10):
            qualified_for_top30.append(model)

    if len(qualified_for_top30) < 30:
        print(f"\n⚠️  严格条件下只有{len(qualified_for_top30)}个模型合格，放宽筛选条件...")
        qualified_for_top30 = []
        for model in top100_with_true_test:
            if (abs(model['true_gap']) < 0.25 and
                model['std_test_r2_true'] < 0.12):
                qualified_for_top30.append(model)

    if len(qualified_for_top30) < 30:
        print(f"\n⚠️  放宽条件后仍只有{len(qualified_for_top30)}个，直接从Top100选择...")
        qualified_for_top30 = top100_with_true_test[:min(50, len(top100_with_true_test))]

    qualified_for_top30.sort(key=lambda x: x['mean_test_r2_true'], reverse=True)

    top30_models = qualified_for_top30[:min(30, len(qualified_for_top30))]

    for model in top30_models:
        composite_score = (0.7 * model['mean_test_r2_true'] -
                          0.2 * abs(model['true_gap']) -
                          0.1 * model['std_test_r2_true'])
        model['composite_score'] = composite_score

    print(f"\n✅ Top{len(top30_models)}模型已选出")
    print(f"\nTop{len(top30_models)}预览:")
    print(f"{'='*80}")

    for rank, model in enumerate(top30_models, 1):
        print(f"{rank:>2}. {model['model_key']:<50} | "
              f"Test R²={model['mean_test_r2_true']:.4f}±{model['std_test_r2_true']:.4f} | "
              f"Gap={model['true_gap']:.4f} | "
              f"综合分={model['composite_score']:.4f}")

    return top100_with_true_test, top30_models, timestamp


# ✅ v2.2修改：save_top30_models_detailed 新增训练集MAE和RMSE的打印和保存
def save_top30_models_detailed(top30_models, timestamp):
    """保存Top30模型详细信息"""
    print("\n" + "="*80)
    print("【保存Top30模型详细信息】")
    print("="*80)

    top30_summary = []

    for rank, model_info in enumerate(top30_models, 1):
        model_key = model_info['model_key']

        print(f"\n{'='*80}")
        print(f"处理第{rank}名: {model_key}")
        print(f"{'='*80}")

        print(f"  🔥 随机化策略: Random State (树模型标准方法)")

        print(f"\n  📋 特征信息:")
        print(f"    特征数量: {len(model_info['features'])}个")
        print(f"    特征列表: {', '.join(model_info['features'])}")

        # ✅ v2.2修改：打印中新增训练集MAE和RMSE
        print(f"\n  📊 训练集性能（留出验证）:")
        print(f"    Train R²   = {model_info['mean_train_r2_training']:.4f} ± {model_info['std_train_r2_training']:.4f}")
        print(f"    Train MAE  = {model_info['mean_train_mae_training']:.4f} ± {model_info['std_train_mae_training']:.4f}")   # ✅ v2.2新增
        print(f"    Train RMSE = {model_info['mean_train_rmse_training']:.4f} ± {model_info['std_train_rmse_training']:.4f}") # ✅ v2.2新增
        print(f"    CV R²      = {model_info['mean_cv_r2_training']:.4f} ± {model_info['std_cv_r2_training']:.4f}")
        print(f"    Test R²    = {model_info['mean_test_r2_training']:.4f} ± {model_info['std_test_r2_training']:.4f}")
        print(f"    Gap        = {model_info['mean_gap_training']:.4f}")

        # ✅ v2.2修改：打印中新增训练集MAE和RMSE（真实测试集部分）
        print(f"\n  📊 真实独立测试集性能:")
        print(f"    Train R²   = {model_info['mean_train_r2_true']:.4f} ± {model_info['std_train_r2_true']:.4f}")
        print(f"    Train MAE  = {model_info['mean_train_mae_true']:.4f} ± {model_info['std_train_mae_true']:.4f}")   # ✅ v2.2新增
        print(f"    Train RMSE = {model_info['mean_train_rmse_true']:.4f} ± {model_info['std_train_rmse_true']:.4f}") # ✅ v2.2新增
        print(f"    Test R²    = {model_info['mean_test_r2_true']:.4f} ± {model_info['std_test_r2_true']:.4f}")
        print(f"    Test MAE   = {model_info['mean_test_mae_true']:.4f} ± {model_info['std_test_mae_true']:.4f}")
        print(f"    Test RMSE  = {model_info['mean_test_rmse_true']:.4f} ± {model_info['std_test_rmse_true']:.4f}")
        print(f"    True Gap   = {model_info['true_gap']:.4f}")

        print(f"\n  📊 综合评估:")
        print(f"    综合得分 = {model_info['composite_score']:.4f}")

        # 保存模型对象（✅ v2.2新增训练集MAE和RMSE字段）
        model_pkl_filename = f"rank{rank:02d}_{model_key}_model.pkl"
        model_pkl_filepath = os.path.join(top30_models_dir, model_pkl_filename)
        with open(model_pkl_filepath, 'wb') as f:
            pickle.dump({
                'model': model_info['model_instance'],
                'scaler': model_info.get('scaler', None),
                'features': model_info['features'],
                'model_name': model_info['model'],
                'group': model_info['group'],
                'k': model_info['k'],

                # 训练集留出验证性能
                'mean_cv_r2_training': model_info['mean_cv_r2_training'],
                'std_cv_r2_training': model_info['std_cv_r2_training'],
                'mean_train_r2_training': model_info['mean_train_r2_training'],
                'std_train_r2_training': model_info['std_train_r2_training'],
                'mean_train_mae_training': model_info['mean_train_mae_training'],    # ✅ v2.2新增
                'std_train_mae_training': model_info['std_train_mae_training'],      # ✅ v2.2新增
                'mean_train_rmse_training': model_info['mean_train_rmse_training'],  # ✅ v2.2新增
                'std_train_rmse_training': model_info['std_train_rmse_training'],    # ✅ v2.2新增
                'mean_test_r2_training': model_info['mean_test_r2_training'],
                'std_test_r2_training': model_info['std_test_r2_training'],
                'mean_gap_training': model_info['mean_gap_training'],

                # 真实测试集性能
                'mean_train_r2_true': model_info['mean_train_r2_true'],
                'std_train_r2_true': model_info['std_train_r2_true'],
                'mean_train_mae_true': model_info['mean_train_mae_true'],    # ✅ v2.2新增
                'std_train_mae_true': model_info['std_train_mae_true'],      # ✅ v2.2新增
                'mean_train_rmse_true': model_info['mean_train_rmse_true'],  # ✅ v2.2新增
                'std_train_rmse_true': model_info['std_train_rmse_true'],    # ✅ v2.2新增
                'mean_test_r2_true': model_info['mean_test_r2_true'],
                'std_test_r2_true': model_info['std_test_r2_true'],
                'mean_test_mae_true': model_info['mean_test_mae_true'],
                'std_test_mae_true': model_info['std_test_mae_true'],
                'mean_test_rmse_true': model_info['mean_test_rmse_true'],
                'std_test_rmse_true': model_info['std_test_rmse_true'],
                'true_gap': model_info['true_gap'],

                'composite_score': model_info['composite_score'],
                'randomization_strategy': 'random_state'
            }, f)

        print(f"\n  💾 文件保存:")
        print(f"    模型文件: {model_pkl_filename}")

        # ✅ v2.2修改：top30_summary中新增训练集MAE和RMSE字段
        top30_summary.append({
            '最终排名': rank,
            '训练排名': model_info.get('rank_training', '-'),
            '模型标识': model_key,
            '组别': model_info['group'],
            'K值': model_info['k'],
            '模型类型': model_info['model'],
            '随机化策略': 'Random State',
            '综合得分': f"{model_info['composite_score']:.6f}",

            # 训练集留出验证性能
            'Train_R²_Training': f"{model_info['mean_train_r2_training']:.6f}",
            'Std_Train_R²_Training': f"{model_info['std_train_r2_training']:.6f}",
            'Train_MAE_Training': f"{model_info['mean_train_mae_training']:.6f}",    # ✅ v2.2新增
            'Std_Train_MAE_Training': f"{model_info['std_train_mae_training']:.6f}", # ✅ v2.2新增
            'Train_RMSE_Training': f"{model_info['mean_train_rmse_training']:.6f}",  # ✅ v2.2新增
            'Std_Train_RMSE_Training': f"{model_info['std_train_rmse_training']:.6f}",# ✅ v2.2新增
            'CV_R²_Training': f"{model_info['mean_cv_r2_training']:.6f}",
            'Std_CV_R²_Training': f"{model_info['std_cv_r2_training']:.6f}",
            'Test_R²_Training': f"{model_info['mean_test_r2_training']:.6f}",
            'Std_Test_R²_Training': f"{model_info['std_test_r2_training']:.6f}",
            'Gap_Training': f"{model_info['mean_gap_training']:.6f}",

            # 真实测试集性能
            'Train_R²_True': f"{model_info['mean_train_r2_true']:.6f}",
            'Std_Train_R²_True': f"{model_info['std_train_r2_true']:.6f}",
            'Train_MAE_True': f"{model_info['mean_train_mae_true']:.6f}",    # ✅ v2.2新增
            'Std_Train_MAE_True': f"{model_info['std_train_mae_true']:.6f}", # ✅ v2.2新增
            'Train_RMSE_True': f"{model_info['mean_train_rmse_true']:.6f}",  # ✅ v2.2新增
            'Std_Train_RMSE_True': f"{model_info['std_train_rmse_true']:.6f}",# ✅ v2.2新增
            'Test_R²_True': f"{model_info['mean_test_r2_true']:.6f}",
            'Std_Test_R²_True': f"{model_info['std_test_r2_true']:.6f}",
            'Test_MAE_True': f"{model_info['mean_test_mae_true']:.6f}",
            'Std_Test_MAE_True': f"{model_info['std_test_mae_true']:.6f}",
            'Test_RMSE_True': f"{model_info['mean_test_rmse_true']:.6f}",
            'Std_Test_RMSE_True': f"{model_info['std_test_rmse_true']:.6f}",
            'Gap_True': f"{model_info['true_gap']:.6f}",

            '特征数量': len(model_info['features']),
            '特征列表': ', '.join(model_info['features'])
        })

    # ✅ v2.2修改：Top30 Excel中新增训练集MAE和RMSE列
    top30_df = pd.DataFrame(top30_summary)
    excel_path = os.path.join(top30_dir, f'top30_models_summary_{timestamp}.xlsx')

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Sheet1: 核心性能（✅ v2.2新增训练集MAE和RMSE）
        top30_df[['最终排名', '训练排名', '模型标识', '组别', 'K值', '模型类型',
                 '综合得分',
                 'Train_R²_Training', 'Train_MAE_Training', 'Train_RMSE_Training',
                 'CV_R²_Training', 'Test_R²_Training', 'Gap_Training',
                 'Train_R²_True', 'Train_MAE_True', 'Train_RMSE_True',
                 'Test_R²_True', 'Gap_True']].to_excel(
            writer, sheet_name='核心性能对比', index=False
        )
        # Sheet2: 特征信息
        top30_df[['最终排名', '模型标识', '特征数量', '特征列表']].to_excel(
            writer, sheet_name='特征信息', index=False
        )
        # Sheet3: 完整信息
        top30_df.to_excel(writer, sheet_name='完整信息', index=False)

    print(f"\n✅ Top{len(top30_models)}汇总Excel已保存: {excel_path}")

    return excel_path


# ==================== 阶段8：排序一致性验证 ====================
def evaluate_model_on_full_data_cv(model_info, X_full, y_full, features_60):
    """在全部数据上进行交叉验证评估单个树模型"""
    model_key = model_info['model_key']
    model_features = model_info['features']

    feature_indices = [features_60.index(feat) for feat in model_features]
    X_model = X_full[:, feature_indices]

    need_scaling = False
    cv_config = CONFIG['cv_strategy']['stage8']

    model_name = model_info['model']

    if 'RandomForest' in model_name:
        base_model = RandomForestRegressor(
            n_estimators=150, max_depth=6, min_samples_split=8,
            min_samples_leaf=4, max_features='sqrt',
            random_state=CONFIG['random_state'], n_jobs=1
        )
    elif 'XGBoost' in model_name:
        base_model = XGBRegressor(
            n_estimators=150, max_depth=4, learning_rate=0.08,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0,
            random_state=CONFIG['random_state'], n_jobs=1, verbosity=0
        )
    elif 'GradientBoosting' in model_name:
        base_model = GradientBoostingRegressor(
            n_estimators=120, max_depth=3, learning_rate=0.08,
            subsample=0.8, min_samples_split=8, min_samples_leaf=4,
            random_state=CONFIG['random_state']
        )
    elif 'ExtraTrees' in model_name:
        base_model = ExtraTreesRegressor(
            n_estimators=200, max_depth=10, min_samples_split=5,
            random_state=CONFIG['random_state'], n_jobs=1
        )
    elif 'HistGradientBoosting' in model_name:
        base_model = HistGradientBoostingRegressor(
            max_iter=150, max_depth=8, learning_rate=0.05,
            random_state=CONFIG['random_state']
        )
    else:
        base_model = model_info['model_instance']

    try:
        if cv_config['method'] == 'repeated_kfold':
            cv_splitter = RepeatedKFold(
                n_splits=cv_config['n_splits'],
                n_repeats=cv_config['n_repeats'],
                random_state=cv_config['random_state']
            )
        else:
            cv_splitter = KFold(
                n_splits=cv_config['n_splits'],
                shuffle=cv_config.get('shuffle', True),
                random_state=cv_config['random_state']
            )

        cv_scores = []

        for fold_idx, (train_idx, val_idx) in enumerate(cv_splitter.split(X_model)):
            X_train_fold = X_model[train_idx]
            X_val_fold = X_model[val_idx]
            y_train_fold = y_full[train_idx]
            y_val_fold = y_full[val_idx]

            model = clone(base_model)
            seed = CONFIG['random_state'] + fold_idx

            valid_params = model.get_params().keys()
            if 'random_state' in valid_params:
                try:
                    model.set_params(random_state=seed)
                except (ValueError, TypeError):
                    pass

            try:
                model.fit(X_train_fold, y_train_fold)
                val_pred = model.predict(X_val_fold)
                val_score = r2_score(y_val_fold, val_pred)
                cv_scores.append(val_score)
            except Exception as e:
                continue

        if len(cv_scores) > 0:
            return np.mean(cv_scores)
        else:
            return None

    except Exception as e:
        print(f"  ⚠️  {model_key} 全数据CV失败: {str(e)}")
        return None


def stage8_ranking_consistency_verification(top30_models, X_full, y_full, features_60, timestamp):
    """阶段8：排序一致性验证（树模型组）"""
    print("\n" + "="*80)
    print("【阶段8】排序一致性验证 - 核心创新（树模型组）")
    print("="*80)

    print(f"\n目的: 验证Top{len(top30_models)}在全数据上的排序稳定性")
    print(f"输入: Top{len(top30_models)}模型架构 + 全部{X_full.shape[0]}样本")
    print(f"方法: {CONFIG['cv_strategy']['stage8']['method']}交叉验证（{CONFIG['cv_strategy']['stage8']['n_splits']}折）")

    print(f"\n开始在全数据（{X_full.shape[0]}样本）上交叉验证Top{len(top30_models)}模型...")

    start_time = time.time()

    cv_full_results = Parallel(n_jobs=CONFIG['stage6_n_jobs'], backend='loky', verbose=10)(
        delayed(evaluate_model_on_full_data_cv)(
            model_info, X_full, y_full, features_60
        ) for model_info in top30_models
    )

    elapsed_time = time.time() - start_time

    print(f"\n✅ 全数据交叉验证完成")
    print(f"⏱️  耗时: {elapsed_time/60:.1f}分钟")

    ranking_comparison = []

    for i, model_info in enumerate(top30_models):
        cv_full_score = cv_full_results[i]

        if cv_full_score is None:
            cv_full_score = -np.inf

        ranking_comparison.append({
            'model_key': model_info['model_key'],
            'group': model_info['group'],
            'model': model_info['model'],
            'k': model_info['k'],
            'features': model_info['features'],
            'rank_test_subset': i + 1,
            'test_r2_subset': model_info['mean_test_r2_true'],
            'cv_r2_full': cv_full_score,
            'randomization_strategy': 'random_state'
        })

    ranking_comparison_sorted = sorted(ranking_comparison,
                                      key=lambda x: x['cv_r2_full'],
                                      reverse=True)

    for new_rank, item in enumerate(ranking_comparison_sorted, 1):
        item['rank_cv_full'] = new_rank

    ranking_comparison_sorted_by_test = sorted(ranking_comparison,
                                              key=lambda x: x['rank_test_subset'])

    for item in ranking_comparison_sorted_by_test:
        model_key = item['model_key']
        for sorted_item in ranking_comparison_sorted:
            if sorted_item['model_key'] == model_key:
                item['rank_cv_full'] = sorted_item['rank_cv_full']
                break

        item['rank_change'] = abs(item['rank_test_subset'] - item['rank_cv_full'])
        item['rank_change_pct'] = (item['rank_change'] / len(top30_models)) * 100

    from scipy.stats import spearmanr

    ranks_test = [item['rank_test_subset'] for item in ranking_comparison_sorted_by_test]
    ranks_cv_full = [item['rank_cv_full'] for item in ranking_comparison_sorted_by_test]

    spearman_rho, spearman_p = spearmanr(ranks_test, ranks_cv_full)

    print(f"\n" + "="*80)
    print("📊 排序一致性分析结果（树模型组）")
    print("="*80)

    print(f"\n🔥 核心指标:")
    print(f"  Spearman相关系数 ρ = {spearman_rho:.4f} (p={spearman_p:.6f})")

    if spearman_rho > 0.85:
        consistency_rating = "A级 - 高度一致"
        consistency_conclusion = "✅ 排序稳定，确认Top30"
    elif spearman_rho > 0.70:
        consistency_rating = "B级 - 部分一致"
        consistency_conclusion = "⚠️  排序部分一致，需深度分析"
    else:
        consistency_rating = "C级 - 不稳定"
        consistency_conclusion = "❌ 排序不稳定，需重新评估"

    print(f"  排序一致性评级: {consistency_rating}")
    print(f"  结论: {consistency_conclusion}")

    avg_rank_change = np.mean([item['rank_change'] for item in ranking_comparison_sorted_by_test])
    print(f"\n  平均排名变化: Δ_Rank = {avg_rank_change:.2f}")

    highly_stable_count = sum(1 for item in ranking_comparison_sorted_by_test if item['rank_change'] <= 3)
    print(f"  高度稳定模型数: {highly_stable_count}/{len(top30_models)} (Δ_Rank≤3)")

    print(f"\n📋 Top{len(top30_models)}排名对比详表:")
    print(f"{'='*100}")
    print(f"{'测试集排名':>10} | {'全数据CV排名':>15} | {'排名变化':>10} | {'模型标识':<50}")
    print(f"{'-'*100}")

    for item in ranking_comparison_sorted_by_test:
        change_symbol = ""
        if item['rank_change'] == 0:
            change_symbol = "="
        elif item['rank_cv_full'] < item['rank_test_subset']:
            change_symbol = "↑"
        else:
            change_symbol = "↓"

        print(f"{item['rank_test_subset']:>10} | {item['rank_cv_full']:>15} | "
              f"{change_symbol}{item['rank_change']:>9} | {item['model_key']:<50}")

    unstable_models = [item for item in ranking_comparison_sorted_by_test
                      if item['rank_change'] > 5]

    if len(unstable_models) > 0:
        print(f"\n⚠️  排名变化较大的模型 (Δ_Rank > 5):")
        print(f"{'='*100}")
        for item in unstable_models:
            print(f"  {item['model_key']:<50} | "
                  f"测试集排名={item['rank_test_subset']:>2} → "
                  f"全数据CV排名={item['rank_cv_full']:>2} | "
                  f"变化={item['rank_change']}")
    else:
        print(f"\n✅ 所有模型排名变化 ≤ 5，稳定性良好")

    ranking_data = []

    for item in ranking_comparison_sorted_by_test:
        ranking_data.append({
            '测试集排名': item['rank_test_subset'],
            '全数据CV排名': item['rank_cv_full'],
            '排名变化量': item['rank_change'],
            '排名变化百分比': f"{item['rank_change_pct']:.1f}%",
            '模型标识': item['model_key'],
            '组别': item['group'],
            '模型类型': item['model'],
            'K值': item['k'],
            '随机化策略': 'Random State',
            'Test_R²_测试集': f"{item['test_r2_subset']:.6f}",
            'CV_R²_全数据': f"{item['cv_r2_full']:.6f}",
            '特征数量': len(item['features']),
            '特征列表': ', '.join(item['features'])
        })

    ranking_df = pd.DataFrame(ranking_data)

    excel_path = os.path.join(stage8_dir, f'ranking_consistency_analysis_{timestamp}.xlsx')

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        ranking_df[['测试集排名', '全数据CV排名', '排名变化量', '排名变化百分比',
                   '模型标识', '组别', '模型类型', 'K值',
                   'Test_R²_测试集', 'CV_R²_全数据']].to_excel(
            writer, sheet_name='排序对比总表', index=False
        )

        correlation_data = [{
            'Spearman_ρ': spearman_rho,
            'p值': spearman_p,
            '平均排名变化': avg_rank_change,
            '高度稳定模型数': highly_stable_count,
            '一致性评级': consistency_rating,
            '结论': consistency_conclusion
        }]
        correlation_df = pd.DataFrame(correlation_data)
        correlation_df.to_excel(writer, sheet_name='Spearman相关系数', index=False)

        if len(unstable_models) > 0:
            unstable_df = ranking_df[ranking_df['排名变化量'] > 5]
            unstable_df.to_excel(writer, sheet_name='排名不稳定模型', index=False)

        ranking_df.to_excel(writer, sheet_name='完整信息', index=False)

    print(f"\n✅ 排序一致性分析结果已保存: {excel_path}")

    return {
        'spearman_rho': spearman_rho,
        'spearman_p': spearman_p,
        'avg_rank_change': avg_rank_change,
        'highly_stable_count': highly_stable_count,
        'consistency_rating': consistency_rating,
        'consistency_conclusion': consistency_conclusion,
        'ranking_comparison': ranking_comparison_sorted_by_test,
        'excel_path': excel_path
    }


# ==================== 主程序 ====================
if __name__ == "__main__":
    print("="*80)
    print("程序1：树模型组完整流程（阶段0-8）- ✅ 优化版 v2.2")
    print("="*80)
    print("\n专注模型：RandomForest / XGBoost / GradientBoosting / ExtraTrees / HistGB")
    print("\n✅ 主要优化:")
    print("  1. 放宽筛选阈值（min_r2=0.45, gap=0.35, penalty=0.08）")
    print("  2. CV策略改为标准KFold（降低标准差，提升效率）")
    print("  3. 减少随机种子（10→5）")
    print("  4. 优化RF超参数（max_depth 10→6，增加正则化）")
    print("  5. 优化XGB超参数（max_depth 6→4，learning_rate 0.05→0.08）")
    print("  6. 优化GB超参数（max_depth 5→3，learning_rate 0.05→0.08）")
    print("  7. 删除非树模型导入")
    print("  8. ✅ v2.2新增：训练集MAE和RMSE的计算、打印和保存")
    print("="*80)

    # ==================== 阶段0：数据准备 ====================
    print("\n【阶段0】数据准备")
    df = pd.read_excel(CONFIG['data_file'])
    print(f"成功加载数据: {df.shape}")

    target_col = CONFIG['target_col']
    exclude_cols = CONFIG['exclude_cols']
    all_columns = df.columns.tolist()

    print("\n步骤0.1：识别原始数值特征")
    features_name = []
    for col in all_columns:
        if col != target_col and col not in exclude_cols:
            if pd.api.types.is_numeric_dtype(df[col]):
                features_name.append(col)

    print(f"  识别到 {len(features_name)} 个原始数值特征")

    print("\n步骤0.2：提取原始特征数据")
    X_all = df[features_name].values
    y_all = df[target_col].values.ravel()
    print(f"  原始数据形状: X{X_all.shape}, y{y_all.shape}")

    print("\n步骤0.3：✅ 检查并移除 NaN 和 Inf/-Inf")
    X_all, y_all, valid_indices = check_and_remove_invalid_values(
        X_all, y_all, "原始数据"
    )
    print(f"  清洗后数据: X{X_all.shape}, y{y_all.shape}")

    df_clean = df.iloc[valid_indices].reset_index(drop=True)

    print("\n步骤0.4：分层采样数据分割")
    stratify_labels = create_stratify_bins(y_all, n_bins=5)

    print("\n步骤0.5：划分训练/测试集")
    X_train, X_test, y_train, y_test, train_indices, test_indices = train_test_split(
        X_all, y_all, np.arange(len(y_all)),
        test_size=CONFIG['test_size'],
        stratify=stratify_labels,
        random_state=CONFIG['random_state']
    )

    print(f"  训练集: {X_train.shape[0]} 样本 × {X_train.shape[1]} 特征")
    print(f"  测试集: {X_test.shape[0]} 样本 × {X_test.shape[1]} 特征")

    print("\n步骤0.6：特征分级标记")
    feature_tier_map = assign_feature_tiers(features_name)

    print("\n" + "="*80)
    print("✅ 数据准备完成")
    print("="*80)
    print("=== 测试集封存，阶段1-6仅在训练集上操作 ===")
    print("="*80)

    # ==================== 阶段1：快速粗筛 ====================
    stage1_keep_indices, stage1_features = stage1_quick_filter(
        X_train, y_train, features_name, feature_tier_map
    )
    X_train_stage1 = X_train[:, stage1_keep_indices]
    X_test_stage1 = X_test[:, stage1_keep_indices]
    stage1_feature_tier_map = {feat: feature_tier_map[feat] for feat in stage1_features}

    # ==================== 阶段2：综合筛选 ====================
    stage2_keep_indices, stage2_features = stage2_integrated_selection(
        X_train_stage1, y_train, stage1_features, stage1_feature_tier_map
    )
    X_train_stage2 = X_train_stage1[:, stage2_keep_indices]
    X_test_stage2 = X_test_stage1[:, stage2_keep_indices]
    stage2_feature_tier_map = {feat: stage1_feature_tier_map[feat] for feat in stage2_features}

    print("\n" + "="*80)
    print("✅ 特征筛选完成（阶段1-2）")
    print("="*80)
    print(f"最终进入模型训练: {len(stage2_features)}个特征")

    # ==================== 阶段4：树模型组特征选择 ====================
    tree_group_data = stage4_tree_model_selection(
        X_train_stage2, y_train, stage2_features, stage2_feature_tier_map
    )

    # ==================== 阶段5：树模型组穷举搜索 ====================
    tree_group_indices = tree_group_data['indices']
    tree_group_features = tree_group_data['features']
    X_train_tree = X_train_stage2[:, tree_group_indices]

    tree_combinations = stage5_tree_group_exhaustive_search(
        X_train_tree, y_train, tree_group_features
    )

    # ==================== 阶段6：树模型组最终训练 ====================
    all_results, all_qualified_models, timestamp = stage6_tree_group_final_training(
        X_train_stage2, X_test_stage2, y_train, y_test,
        train_indices, test_indices, tree_combinations, tree_group_indices
    )

    # ==================== 阶段7：结果输出与排序 ====================
    print("\n" + "="*80)
    print("【阶段7】结果输出、Top100评估与Top30筛选")
    print("="*80)

    # ✅ v2.2修改：all_results_df中包含训练集MAE和RMSE列
    all_results_df = pd.DataFrame(all_results)
    csv_path = os.path.join(CONFIG['output_dir'], f'all_model_results_{timestamp}.csv')
    all_results_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"\n所有模型结果CSV已保存: {csv_path}")

    if all_qualified_models:
        print(f"\n原始合格模型数: {len(all_qualified_models)}个")

        all_qualified_models_sorted = sort_models_by_test_r2_with_stability(all_qualified_models)

        top100_with_true_test, top30_models, timestamp = stage7_evaluate_top_models_on_true_test(
            all_qualified_models_sorted, X_train_stage2, X_test_stage2,
            y_train, y_test, stage2_features, timestamp
        )

        top30_excel_path = save_top30_models_detailed(top30_models, timestamp)

        model_counter = Counter([m['model'] for m in all_qualified_models])

        print(f"\n" + "="*80)
        print("📊 按模型类型统计合格模型:")
        print("="*80)
        for model_name in ['RandomForest', 'XGBoost', 'GradientBoosting', 'ExtraTrees', 'HistGradientBoosting']:
            count = model_counter.get(model_name, 0)
            print(f"  {model_name:25s}: {count:3d}个")

        # ==================== 阶段8：排序一致性验证 ====================
        print("\n" + "="*80)
        print("准备进入阶段8 - 排序一致性验证...")
        print("="*80)

        X_full_stage2 = np.vstack([X_train_stage2, X_test_stage2])
        y_full = np.concatenate([y_train, y_test])

        print(f"\n全数据信息:")
        print(f"  总样本数: {X_full_stage2.shape[0]}")
        print(f"  特征数: {X_full_stage2.shape[1]}")

        stage8_results = stage8_ranking_consistency_verification(
            top30_models, X_full_stage2, y_full, stage2_features, timestamp
        )

        print(f"\n" + "="*80)
        print("📁 输出文件汇总:")
        print("="*80)
        print(f"1. 所有模型结果CSV: {csv_path}")
        print(f"2. Top100真实测试性能Excel: {os.path.join(top100_dir, f'top100_true_test_performance_{timestamp}.xlsx')}")
        print(f"3. Top30模型汇总Excel: {top30_excel_path}")
        print(f"4. Top30模型对象: {top30_models_dir}")
        print(f"5. 🔥 排序一致性分析Excel: {stage8_results['excel_path']}")

        # ==================== 最终总结 ====================
        print(f"\n" + "="*80)
        print("✅ 程序1执行完成! (树模型组 - 优化版 v2.2)")
        print("="*80)

        print(f"\n" + "="*80)
        print("🎯 关键成果汇总:")
        print("="*80)
        print(f"  • 合格模型总数: {len(all_qualified_models)}个")
        print(f"  • Top{len(top100_with_true_test)}模型已在真实测试集评估")
        print(f"  • Top{len(top30_models)}模型已确认")
        print(f"  • 排序一致性验证完成:")
        print(f"     - Spearman ρ = {stage8_results['spearman_rho']:.4f}")
        print(f"     - 评级: {stage8_results['consistency_rating']}")
        print(f"     - 结论: {stage8_results['consistency_conclusion']}")
        print(f"     - 高度稳定模型: {stage8_results['highly_stable_count']}/{len(top30_models)}")
        print(f"     - 平均排名变化: {stage8_results['avg_rank_change']:.2f}")

        print(f"\n" + "="*80)
        print("🔬 树模型组模型表现:")
        print("="*80)
        top30_by_model = {}
        for model in top30_models:
            model_type = model['model']
            if model_type not in top30_by_model:
                top30_by_model[model_type] = 0
            top30_by_model[model_type] += 1

        for model_name in ['RandomForest', 'XGBoost', 'GradientBoosting', 'ExtraTrees', 'HistGradientBoosting']:
            count = top30_by_model.get(model_name, 0)
            print(f"  {model_name:25s}在Top{len(top30_models)}中: {count:2d}个")

        print(f"\n" + "="*80)
        print("🚀 下一步:")
        print("="*80)
        print(f"  使用Top{len(top30_models)}模型进行主动学习")
        print(f"  基于不确定性采样策略")
        print(f"  迭代优化模型性能")
        print("="*80)

    else:
        print("\n⚠️  无满足条件的模型")

    print(f"\n" + "="*80)
    print("✅ 程序1完整执行完毕! (树模型组 - 优化版 v2.2)")
    print("="*80)

### ✅ 程序1_树模型组_完整流程_优化版v2.2.py 完成 ###

检测到 12 个CPU核心
程序1：树模型组完整流程（阶段0-8）- ✅ 优化版 v2.2

专注模型：RandomForest / XGBoost / GradientBoosting / ExtraTrees / HistGB

✅ 主要优化:
  1. 放宽筛选阈值（min_r2=0.45, gap=0.35, penalty=0.08）
  2. CV策略改为标准KFold（降低标准差，提升效率）
  3. 减少随机种子（10→5）
  4. 优化RF超参数（max_depth 10→6，增加正则化）
  5. 优化XGB超参数（max_depth 6→4，learning_rate 0.05→0.08）
  6. 优化GB超参数（max_depth 5→3，learning_rate 0.05→0.08）
  7. 删除非树模型导入
  8. ✅ v2.2新增：训练集MAE和RMSE的计算、打印和保存

【阶段0】数据准备
成功加载数据: (202, 209)

步骤0.1：识别原始数值特征
  识别到 208 个原始数值特征

步骤0.2：提取原始特征数据
  原始数据形状: X(202, 208), y(202,)

步骤0.3：✅ 检查并移除 NaN 和 Inf/-Inf

  检查原始数据中的无效值...
    ✅ 数据clean，无无效值
  清洗后数据: X(202, 208), y(202,)

步骤0.4：分层采样数据分割

  创建分层标签(5个区间):
    区间 1: KQ范围 [3.40, 7.55], 样本数=41, 均值=6.59
    区间 2: KQ范围 [7.60, 9.59], 样本数=40, 均值=8.50
    区间 3: KQ范围 [9.60, 11.27], 样本数=40, 均值=10.51
    区间 4: KQ范围 [11.38, 13.47], 样本数=40, 均值=12.35
    区间 5: KQ范围 [13.49, 19.80], 样本数=41, 均值=15.15

步骤0.5：划分训练/测试集
  训练集: 161 样本 × 208 特征
  测试集: 41 样本 × 208 特征

步骤0.6：特征分级标记

==================== 特征分级标记 ==========

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:    2.0s
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:   10.3s
[Parallel(n_jobs=6)]: Done  13 tasks      | elapsed:   17.8s
[Parallel(n_jobs=6)]: Done  20 tasks      | elapsed:   27.5s
[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:   37.8s
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:   50.4s
[Parallel(n_jobs=6)]: Done  49 tasks      | elapsed:  1.1min
[Parallel(n_jobs=6)]: Done  60 tasks      | elapsed:  1.3min
[Parallel(n_jobs=6)]: Done  73 tasks      | elapsed:  1.6min
[Parallel(n_jobs=6)]: Done  86 tasks      | elapsed:  1.9min
[Parallel(n_jobs=6)]: Done 100 out of 100 | elapsed:  2.2min finished
[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.



🚀 阶段6并行评估完成统计（✅ 优化版）:
总评估任务数: 100
有效评估数: 100
合格模型数: 41
⏱️  总耗时: 2.2分钟
⚡ 平均每任务: 1.3秒

📊 按模型类型统计合格模型:
  RandomForest             :  17个
  XGBoost                  :   0个
  GradientBoosting         :   7个
  ExtraTrees               :   7个
  HistGradientBoosting     :  10个

【阶段7】结果输出、Top100评估与Top30筛选

所有模型结果CSV已保存: D:\博一下\第一章主动学习\树模型组_结果11.9_优化版-4.22\all_model_results_20260422_234558.csv

原始合格模型数: 41个

【模型排序】按测试集R²均值排序（含稳定性惩罚）

排序策略:
  排序得分 = Mean_Test_R² - 0.1 × Std_Test_R²

✅ 排序完成! Top30模型预览:
 1. Tree_GradientBoosting_K6_Comb13                    | 排序=0.7679 | CV R²=0.5276±0.0105 | Test R²=0.7688±0.0088
 2. Tree_GradientBoosting_K5_Comb7                     | 排序=0.7673 | CV R²=0.5354±0.0103 | Test R²=0.7684±0.0109
 3. Tree_GradientBoosting_K6_Comb15                    | 排序=0.7668 | CV R²=0.5212±0.0144 | Test R²=0.7681±0.0123
 4. Tree_GradientBoosting_K6_Comb14                    | 排序=0.7662 | CV R²=0.5311±0.0064 | Test R²=0.7673±0.0106
 5. Tree_ExtraTrees_K5_Comb7                       

[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:    0.9s
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:    1.9s
[Parallel(n_jobs=6)]: Done  13 tasks      | elapsed:    3.8s
[Parallel(n_jobs=6)]: Done  20 tasks      | elapsed:    5.7s
[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:    7.6s
[Parallel(n_jobs=6)]: Done  35 out of  41 | elapsed:    9.4s remaining:    1.5s
[Parallel(n_jobs=6)]: Done  41 out of  41 | elapsed:   11.0s finished



✅ Top41真实测试集评估完成
⏱️  耗时: 0.2分钟

✅ Top41详细结果已保存: D:\博一下\第一章主动学习\树模型组_结果11.9_优化版-4.22\top100_models\top100_true_test_performance_20260422_234558.xlsx

从Top100中选择Top30（优化的筛选+排序策略）

✅ 综合得分公式:
  方法：筛选 + 排序
    - 先筛选：Gap < 0.20 且 Std < 0.10
    - 再排序：按 Test_R²_True 降序

✅ Top30模型已选出

Top30预览:
 1. Tree_GradientBoosting_K6_Comb13                    | Test R²=0.7688±0.0088 | Gap=0.1519 | 综合分=0.5069
 2. Tree_HistGradientBoosting_K6_Comb13                | Test R²=0.7688±0.0088 | Gap=0.1519 | 综合分=0.5069
 3. Tree_GradientBoosting_K5_Comb7                     | Test R²=0.7684±0.0109 | Gap=0.1534 | 综合分=0.5061
 4. Tree_HistGradientBoosting_K5_Comb7                 | Test R²=0.7684±0.0109 | Gap=0.1534 | 综合分=0.5061
 5. Tree_GradientBoosting_K6_Comb15                    | Test R²=0.7681±0.0123 | Gap=0.1564 | 综合分=0.5051
 6. Tree_HistGradientBoosting_K6_Comb15                | Test R²=0.7681±0.0123 | Gap=0.1564 | 综合分=0.5051
 7. Tree_GradientBoosting_K6_Comb14                    | Test R²=0.7673±0.0106 | G

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:    0.4s
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:    0.4s
[Parallel(n_jobs=6)]: Done  13 tasks      | elapsed:    1.4s
[Parallel(n_jobs=6)]: Done  23 out of  30 | elapsed:    3.1s remaining:    0.9s
[Parallel(n_jobs=6)]: Done  27 out of  30 | elapsed:    3.3s remaining:    0.3s



✅ 全数据交叉验证完成
⏱️  耗时: 0.1分钟

📊 排序一致性分析结果（树模型组）

🔥 核心指标:
  Spearman相关系数 ρ = 0.4376 (p=0.015592)
  排序一致性评级: C级 - 不稳定
  结论: ❌ 排序不稳定，需重新评估

  平均排名变化: Δ_Rank = 7.73
  高度稳定模型数: 5/30 (Δ_Rank≤3)

📋 Top30排名对比详表:
     测试集排名 |         全数据CV排名 |       排名变化 | 模型标识                                              
----------------------------------------------------------------------------------------------------
         1 |              21 | ↓       20 | Tree_GradientBoosting_K6_Comb13                   
         2 |              22 | ↓       20 | Tree_HistGradientBoosting_K6_Comb13               
         3 |               9 | ↓        6 | Tree_GradientBoosting_K5_Comb7                    
         4 |              10 | ↓        6 | Tree_HistGradientBoosting_K5_Comb7                
         5 |              14 | ↓        9 | Tree_GradientBoosting_K6_Comb15                   
         6 |              15 | ↓        9 | Tree_HistGradientBoosting_K6_Comb15               
         7 |              16 | ↓

[Parallel(n_jobs=6)]: Done  30 out of  30 | elapsed:    3.9s finished
